# v8 GQA M-packing on TENSOR CORES — the gate (Cut 2a: Turing WMMA, Colab T4)

Cut 2a is the **GEMV→GEMM step on top of Cut 1's M-packing**: same paged GQA split-KV decode, but the QK and PV matmuls run on **16×16×16 WMMA tensor cores** (v5's fix). `M=G` is padded to 16 (rows ≥G zeroed). Tensor cores DON'T change bytes (`AI=2G/b` unchanged) — the bet is they **close part of Cut 1's per-CTA gap** (Cut 1 left %HBM ≤11%). Runs on the free T4; the A100 `mma.m16n8k16`+cp.async peak version is Cut 2b. **Headline check: does v8_gqa_tc beat v8_gqa (Cut 1)?**

## 0. Dependencies + GPU (venv-safe)

In [ ]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

# 1) Physical GPU on this runtime? (Colab defaults to CPU; pick a GPU explicitly.)
try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit(
        'No GPU on this Colab runtime. FIX: Runtime > Change runtime type > T4 GPU > Save, '
        'then Runtime > Restart session, then re-run from the top. (This kernel needs a Turing T4.)')

# 2) Install deps (incl. numpy) BEFORE importing torch, so torch's numpy bridge initializes --
#    importing torch first on a numpy-less venv (vast.ai) prints 'Failed to initialize NumPy'.
pip('ninja', 'pytest', 'numpy')

# 3) torch present AND CUDA-enabled? A CPU-only wheel raises "not compiled with CUDA" on any kernel.
try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False

if not cuda_ok:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
    raise SystemExit(
        'A GPU is present but torch was a CPU-only build -- installed the CUDA build. NOW: '
        'restart the kernel/session, then re-run this cell (the old CPU torch stays loaded until restart).')

# vast.ai/venv: !-cells spawn a bare shell without the venv on PATH -> `python` not found.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap --format=csv
!which python && python -c "import torch; print('shell python sees torch', torch.__version__)"

## 1. Get the repo

In [ ]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

## 2. Roofline — UNCHANGED from Cut 1 (`AI=2G/b`); tensor cores attack the *schedule*, not the bytes

The model can't see this step: same bytes → same `AI=2G/b`, same HBM floor. The prediction is a schedule claim — the GEMM should lower µs/tok and RAISE %HBM vs Cut 1 at a given G (closing the gap), though at G=8 the WMMA tile is half-empty (8 real rows / 16) so 2a captures the direction, not the peak.

In [ ]:
from roofline.archs import get_arch
from roofline.model import estimate
arch = get_arch('sm_80')
print(f"{'G':>3} | {'AI=2G/b':>8} | {'limiter':>7} | {'t_hbm floor':>12}  (identical to Cut 1 — bytes unchanged)")
for G in (1,2,4,8,16,32):
    e = estimate(arch, B=8, H=8, N_q=1, N_k=8192, d=128, precision='fp16', G=G)
    print(f'{G:>3} | {e.arithmetic_intensity:8.1f} | {e.limiter.upper():>7} | {e.t_hbm*1e3:9.4f}ms')
print('\nPrediction: tensor cores lower us/tok + raise %HBM vs Cut 1 at G>=4 (GEMM closes the per-CTA gap).')


## 3. Build v8_gqa_tc (JIT, Turing WMMA)

In [ ]:
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_v8_gqa_tc')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
from bindings.load import build_kernel
mod = build_kernel('v8_gqa_tc')
print('built:', mod)


## 4. Correctness gate — v8_gqa_tc (Gate 1 of 2)

Same GQA cases as Cut 1 (decode `G∈{1,2,4,8}` × non-multiple `N_k` × causal+offset, idle/pad `G=3` + multi-tile `G=16`, square reduction), now on the WMMA backend. Oracle = `sdpa_reference_gqa`, tol 2e-2.

In [ ]:
!python -m pytest tests/test_correctness.py -k "v8_gqa_tc" -q


## 5. The A/B — v8_gqa_tc (tensor cores) vs v8_gqa (Cut 1, CUDA cores), same G-sweep

Identical workload, both backends. Compare `us/tok` and `%HBM` per G — does the GEMM beat the warp-shuffle GEMV, and by how much as G crosses the M<16→M≥16 (pad→full) line?

In [ ]:
print('=== Cut 2a: v8_gqa_tc (Turing WMMA tensor cores) ===')
!python -m bench.harness --backend v8_gqa_tc --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32
print('\n=== Cut 1: v8_gqa (CUDA cores) — same workload, for the A/B ===')
!python -m bench.harness --backend v8_gqa --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32


## 6. Reclaim-SDPA-at-batch on tensor cores (G=8)

Does the tensor-core path still beat SDPA across the serving batch range (it must at least match Cut 1)?

In [ ]:
!python -m bench.harness --backend v8_gqa_tc --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64
